# W04-02 · 검색과 출처 확인

필수 25분 + 확장 15분. 질문에서 검색된 문서와 실제 인용 위치까지 연결합니다.

**진행 방식**: 먼저 실행 결과를 예측하고 셀을 실행합니다. 변경 실습은 제공된 값 하나를 바꿉니다. 핵심 셀은 네트워크·API 키 없이 실행됩니다. 모든 정책·질의는 교육용 합성 데이터입니다.

VS Code → 우측 상단 Select Kernel → 이 저장소의 .venv (Python 3.11). 노트북별로 Restart 후 Run All 가능합니다.

In [ ]:
from pathlib import Path
import sys, json, math, statistics
from dataclasses import asdict, replace
from collections import Counter
from time import perf_counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course_labs").is_dir()), None)
assert ROOT is not None, "저장소 내부에서 notebook을 열어 주세요."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_labs.retrieval import (
    Document,
    Chunk,
    LexicalIndex,
    load_documents,
    load_cases,
    chunk_document,
    tokenize,
    expand_query,
    eligible_chunks,
    reciprocal_rank_fusion,
    rerank_by_coverage,
    retrieval_metrics,
    retrieve,
    answer_from_evidence,
    evaluate,
)
from IPython.display import display, HTML

docs = load_documents()
cases = load_cases()
print("Python:", sys.version.split()[0], "| 문서:", len(docs), "| 평가 질의:", len(cases))
assert sys.version_info[:2] == (3, 11), "과정 기준 Python 3.11.x 커널 선택"

In [ ]:
def show(rows, columns=None):
    import html

    if not rows:
        return display(HTML("<p>결과 없음</p>"))
    cols = columns or list(rows[0])
    headers = "".join(
        '<th style="padding:8px;text-align:left">' + html.escape(str(c)) + "</th>" for c in cols
    )
    body = "".join(
        "<tr>"
        + "".join(
            '<td style="padding:8px;border-bottom:1px solid #ddd">'
            + html.escape(str(r.get(c, "")))
            + "</td>"
            for c in cols
        )
        + "</tr>"
        for r in rows
    )
    display(
        HTML(
            '<table style="font-size:15px;border-collapse:collapse"><tr>'
            + headers
            + "</tr>"
            + body
            + "</table>"
        )
    )

## 01 첫 검색

정답 문서가 무엇인지 먼저 예상합니다.

In [ ]:
r = retrieve("상품 반품 기간")
show(
    [
        {
            "rank": i + 1,
            "doc": h["chunk"]["doc_id"],
            "title": h["chunk"]["title"],
            "score": round(h["score"], 3),
            "text": h["chunk"]["text"],
        }
        for i, h in enumerate(r["hits"])
    ]
)

**관찰·변경 과제**

반품 기간과 배송비가 다른 문서인 이유를 구분하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 02 질문 표현 변경

같은 의도라도 표현이 바뀌면 키워드 검색이 달라집니다.

In [ ]:
for q in ["상품 반품 기간", "상품 되돌려 보내는 기간", "돈은 언제 돌려줘요"]:
    print(q, [h["chunk"]["doc_id"] for h in retrieve(q)["hits"]])

**관찰·변경 과제**

질의 표현만 바꿔 검색 실패 1개를 만드세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 03 동의어 확장

동의어 사전은 명시적 규칙입니다. 의미 이해 모델과 구분합니다.

In [ ]:
q = "돈은 언제 돌려줘요"
print("확장:", expand_query(q))
for expand in [False, True]:
    print(expand, [h["chunk"]["doc_id"] for h in retrieve(q, expand=expand)["hits"]])

**관찰·변경 과제**

단어 “돈”이 모든 상황에서 환불을 뜻할까요? 잘못된 확장 사례를 적어 보세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 04 원문 발췌 응답

실습의 기본 응답은 LLM 생성이 아니라 검색 원문입니다.

In [ ]:
r = retrieve("상품 반품 기간")
a = answer_from_evidence(r)
print(json.dumps(a, ensure_ascii=False, indent=2))
assert a["mode"] == "extractive-preview"

**관찰·변경 과제**

사용자에게 “생성된 답변”이라고 잘못 소개하면 어떤 혼동이 생길까요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 05 인용 검증

표시된 구간과 원본이 실제로 같은지 검증합니다.

In [ ]:
citation = a["citations"][0]
source_doc = next(d for d in docs if d.id == citation["doc_id"])
assert source_doc.version == citation["version"]
assert source_doc.text[citation["start"] : citation["end"]] == a["text"]
print("문서:", source_doc.title, "버전:", source_doc.version)

**관찰·변경 과제**

원문 내용이 변경되었는데 ID만 같을 때 어떤 검사가 필요한가요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 06 답변 보류

검색 결과가 비어 있지 않아도 질문에 필요한 근거가 부족할 수 있습니다.

In [ ]:
r_unknown = retrieve("화성 우주 정거장 운임")
print("검색 결과 수:", len(r_unknown["hits"]))
print(answer_from_evidence(r_unknown))

**관찰·변경 과제**

무관한 문서가 반환되는 이유와 답변을 보류해야 하는 이유를 분리하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 07 보류 임계값

단어 겹침 비율은 신뢰도나 정답 확률이 아닙니다.

In [ ]:
for threshold in [0.1, 0.25, 0.5, 0.9]:
    a = answer_from_evidence(r_unknown, threshold)
    print(threshold, a["status"], round(a["term_coverage"], 3))

**관찰·변경 과제**

임계값을 높이면 잘못된 답변이 줄어드는 대신 어떤 요청이 더 보류될까요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 08 최신성 비교

현재 정책과 과거 정책의 차이를 확인합니다.

In [ ]:
for day in ["2026-08-15", "2026-09-29"]:
    r = retrieve("상품 반품 기간", as_of=day)
    print(day, answer_from_evidence(r)["text"])

**관찰·변경 과제**

답변 화면에 기준 날짜를 어디에 보여주면 좋을까요?

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 09 권한 공격 질의

내부 코드를 알아도 접근 권한이 생기지 않습니다.

In [ ]:
r = retrieve("내부 승인 티켓 EX-77", role="public", k=8)
assert all(h["chunk"]["doc_id"] != "D09" for h in r["hits"])
assert all("EX-77" not in h["chunk"]["text"] for h in r["hits"])
print(answer_from_evidence(r))

**관찰·변경 과제**

검색 후 필터링과 검색 전 필터링의 로그 노출 위험을 비교하세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 10 한계 기록

정확한 인용이 모든 사실의 정확성을 보장하지는 않습니다.

In [ ]:
r = retrieve("반품 배송비와 환불 처리 기간")
print([h["chunk"]["doc_id"] for h in r["hits"]])
print("기본 발췌:", answer_from_evidence(r)["text"])

**관찰·변경 과제**

복합 질문인데 첫 문서만 발췌하는 한계를 개선 요구사항으로 쓰세요.

기록: 변경 값 / 기대 결과 / 실제 결과 / 해석 / 남은 한계

## 완료 체크

- 설정 하나를 바꾼 비교 결과와 실패 질의 1개
- 출처·문서 버전·기준 날짜를 포함한 설명
- 측정한 것과 측정하지 않은 것의 구분

## 참고

- [RAG 원 논문](https://arxiv.org/abs/2005.11401)
- [Stanford IR: TF-IDF](https://nlp.stanford.edu/IR-book/html/htmledition/term-frequency-and-weighting-1.html)
- [Stanford IR: BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html)
- [Ollama embed API](https://docs.ollama.com/api/embed)

로컬 구현 위치: course_labs/retrieval.py. 형태소 분석·학습된 임베딩·신경망 재정렬을 구현한 것으로 해석하지 않습니다.